In [16]:
n_way= 5
k_shot= 5
q_query= 5

In [ ]:
from pathlib import Path
import random
from typing import Iterator

from torch import Tensor
import torch.nn as nn
from torch.utils.data import IterableDataset
from torchvision.io import decode_image


class EpisodeDataset(IterableDataset):
    def __init__(self, root: str, n_way: int, k_shot: int, q_query: int, transform: nn.Module | None = None) -> None:
        super().__init__()

        self.root = root
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.transform = transform

        # self.data:list[tuple[Path,int]]=[]
        # [(WindowsPath('omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_01.png'), 0)]

        self.character_to_images: dict[str, list[str]] = {}

        self._load()

    def _load(self) -> None:
        root_path = Path(self.root)

        for language_path in root_path.iterdir():
            if not language_path.is_dir():
                continue

            for character_path in language_path.iterdir():
                if not character_path.is_dir():
                    continue

                for image_path in character_path.iterdir():
                    if not image_path.is_file():
                        continue

                    # self.data.append((image_path,class_i))

                    self.character_to_images.setdefault(character_path.as_posix(), []).append(image_path.as_posix())

        # print(self.character_to_images[r'omniglot/python/images_background/Alphabet_of_the_Magi/character01'])

    def _load_image(self, path_str: str) -> Tensor:
        image = decode_image(path_str)

        if self.transform:
            image = self.transform(image)

        image: Tensor = image.float() / 255

        return image

    def __iter__(self) -> Iterator[tuple[list[Tensor], list[int], list[Tensor], list[int]]]:
        selected_character_strs = random.sample(tuple(self.character_to_images.keys()), self.n_way)

        support: list[Tensor] = []
        support_labels: list[int] = []
        query: list[Tensor] = []
        query_labels: list[int] = []

        for i, character in enumerate(selected_character_strs):
            image_strs: list[str] = random.sample(self.character_to_images[character], self.k_shot + self.q_query)

            images = [self._load_image(image_str) for image_str in image_strs]

            support += images[: self.k_shot]
            support_labels += [i] * self.k_shot
            query += images[self.k_shot :]
            query_labels += [i] * self.q_query

        yield support, support_labels, query, query_labels

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels=1, feature_channels=64, embed_dim=64) -> None:
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, feature_channels, 3, padding=1),
            nn.BatchNorm2d(feature_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(feature_channels, feature_channels, 3, padding=1),
            nn.BatchNorm2d(feature_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(feature_channels, feature_channels, 3, padding=1),
            nn.BatchNorm2d(feature_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(feature_channels, embed_dim, 3, padding=1),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)  # 输出 (batch, z_dim, 1, 1)
        )

    def forward(self, x: Tensor):
        x = self.encoder(x)
        return x.view(x.size(0), -1)  # flatten to (batch, z_dim)

In [ ]:
import torch
import torch.nn as nn


class ProtoNetModel(nn.Module):
    def __init__(self, encoder: nn.Module) -> None:
        super().__init__()

        self.encoder: nn.Module = encoder

    def forward(self, support: Tensor, support_labels: Tensor, query: Tensor) -> Tensor:
        """
        support: (n_way*k_shot, 1, H, W)
        support_labels: (n_way*k_shot,)
        query: (n_way*q_query, 1, H, W)

        return:
            logits: (n_way*q_query, n_way)
        """
        # 1. embed support and query
        embed_support: Tensor = self.encoder(support)  # (n_way*k_shot, embed_dim)
        embed_query: Tensor = self.encoder(query)  # (n_way*q_query, embed_dim)

        # 2. compute prototypes
        unique_labels = torch.unique(support_labels)  # [0,1,2,...,n_way-1]
        prototypes_list = []
        for c in unique_labels:
            # mean of embeddings for class c
            embed_class = embed_support[support_labels == c]
            prototypes_list.append(embed_class.mean(0))

        prototypes: Tensor = torch.stack(prototypes_list)  # (n_way, embed_dim)

        # 3. compute negative distances as logits
        # dist: (n_query, n_way)
        dist = torch.cdist(embed_query, prototypes)

        # negative distance = higher score if closer
        logits = -dist

        return logits

In [ ]:
train_set = EpisodeDataset(root='./omniglot/python/images_background', n_way=n_way, k_shot=k_shot, q_query=q_query)
test_set = EpisodeDataset(root='./omniglot/python/images_evaluation', n_way=n_way, k_shot=k_shot, q_query=q_query)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set)
test_loader = DataLoader(test_set)